# 🏃 FitByte AI Content Creator
## Project 2 — AI-Powered Brand Content Generation

**Module 2 | Team Project**

---

### What This Notebook Demonstrates

This notebook walks through the full **FitByte AI Content Creator** — a Python system that generates on-brand blog posts, Instagram captions and LinkedIn posts for FitByte, a precision fitness watch company.

**The pipeline: Document → Monitor → Brief → Publish → Iterate**

| Stage | What happens |
|---|---|
| Document | Load both knowledge bases (brand guidelines, product specs, content examples, market research) |
| Monitor | Analyze topic for brand fit and market relevance using the LLM |
| Brief | Chain-of-thought prompt generates a structured content brief |
| Publish | Few-shot + contextual placement prompt generates the final content |
| Iterate | Human review loop — approve, edit or regenerate |

**Uniqueness strategy:** Unlike generic ChatGPT outputs, every prompt injects FitByte's actual brand voice, writing rules, real product specs, past content examples, and market trends. The final section demonstrates this difference side-by-side.


## 1. Setup & Configuration

In [1]:
import os
import sys
from pathlib import Path

# Add src/ to path so we can import our modules
sys.path.insert(0, str(Path("src").resolve()))

# Install dependencies if needed
# !pip install openai anthropic python-dotenv

from dotenv import load_dotenv
load_dotenv()

print("✓ Environment loaded")
print(f"  OpenAI key set: {'OPENAI_API_KEY' in os.environ and bool(os.environ['OPENAI_API_KEY'])}")
print(f"  Anthropic key set: {'ANTHROPIC_API_KEY' in os.environ and bool(os.environ['ANTHROPIC_API_KEY'])}")


✓ Environment loaded
  OpenAI key set: True
  Anthropic key set: True


## 2. Knowledge Base — Loading Both Sources

The system uses **two knowledge bases** stored as markdown files:

- **Primary KB** (`knowledge_base/primary/`): Brand guidelines, product specs, past content examples
- **Secondary KB** (`knowledge_base/secondary/`): Market trends, competitor analysis

No vector database is required. We load and parse the markdown files, then inject relevant excerpts directly into our LLM prompts.


In [2]:
import re
from document_processor import load_and_clean, strip_markdown
from knowledge_base import KnowledgeBase

kb = KnowledgeBase()

# Load and inspect each component
brand_voice = kb.brand_voice()
writing_rules = kb.writing_rules()
product_specs = kb.product_specs()
content_examples = kb.content_examples(n=3)
market_context = kb.market_context()
differentiators = kb.competitor_context()

print("=== KNOWLEDGE BASE LOADED ===\n")
print(f"Brand voice:        {len(brand_voice):,} chars")
print(f"Writing rules:      {len(writing_rules):,} chars")
print(f"Product specs:      {len(product_specs):,} chars")
print(f"Content examples:   {len(content_examples):,} chars")
print(f"Market context:     {len(market_context):,} chars")
print(f"Differentiators:    {len(differentiators):,} chars")
print()
print("--- Brand Voice (first 400 chars) ---")
print(brand_voice[:400])


=== KNOWLEDGE BASE LOADED ===

Brand voice:        7,812 chars
Writing rules:      0 chars
Product specs:      2,503 chars
Content examples:   2,149 chars
Market context:     1,570 chars
Differentiators:    1,253 chars

--- Brand Voice (first 400 chars) ---
Brand Guidelines
FitByte — Brand Voice & Content Standards

Who We Are

FitByte is a fitness watch built for people who are serious about their health — but don't want a device that makes them feel like they're in a lab.

We sit at the intersection of performance technology and everyday life. Our wearers range from competitive athletes tracking race-day splits to busy professionals who want to clo


In [3]:
# Preview the content examples — these are FitByte's real blog posts
# used as style references in our prompts
print("--- Content Style Examples (first 500 chars) ---")
print(content_examples[:500])


--- Content Style Examples (first 500 chars) ---
Post 01 — Recovery

Rest Days Are Part of the Training

You don't get fitter during your workout. You get fitter after it — while you sleep, recover and let your body do its thing.

That's what your Recovery Score is tracking. When it's low, your body is still rebuilding. Pushing hard anyway doesn't make you tougher. It just slows everything down.

On low-recovery days, go for a walk. Do some stretching. Cook a good meal. The gym will still be there tomorrow — and you'll actually get something o


In [4]:
# Preview market context — feeds the Monitor step and Brief step
print("--- Market Context (first 500 chars) ---")
print(market_context[:500])


--- Market Context (first 500 chars) ---
Market Trends & Industry Context
Fitness Wearables — Research Layer (2024–2025)

Market Overview

The global fitness tracker market is projected to reach $91.3B by 2027. Growth is driven by:
- Rising consumer health awareness post-pandemic
- Employer wellness programs subsidizing wearables
- Integration of clinical-grade sensors (ECG, SpO2) in consumer devices
- Subscription-based coaching and insights models gaining traction

Key Consumer Trends

1. Recovery-First Mindset
- 68% of serious athle


## 3. Prompt Engineering Templates

The prompt templates are the core of our uniqueness strategy. Each template injects:

1. **Role definition** (third-person, as per Module 2 best practices)
2. **Brand voice rules** from the Primary KB
3. **Real content examples** for few-shot learning
4. **Specific product facts** to avoid vague claims
5. **Market context** for positioning

Techniques used: role prompting, few-shot prompting, chain-of-thought, contextual placement.


In [5]:
from prompt_templates import (
    PromptContext, 
    build_brief_prompt, 
    build_blog_post_prompt,
    build_instagram_prompt,
    build_linkedin_prompt,
    SYSTEM_PROMPT_WRITER,
    SYSTEM_PROMPT_ANALYST,
)

# Preview the system prompt (role definition)
print("=== SYSTEM PROMPT (Role Definition) ===")
print(SYSTEM_PROMPT_WRITER)


=== SYSTEM PROMPT (Role Definition) ===
The assistant is an expert content writer for FitByte, a precision fitness watch brand. They have 10 years of experience writing health-tech content that is human, grounded and credible — never generic. They have deeply internalized FitByte's brand voice: clear not clinical, motivating not preachy, precise not verbose, confident not arrogant, human not robotic. They never use passive voice, serial commas, fear-based messaging or unqualified superlatives. Every word earns its place.


In [6]:
# Build a sample context and preview the brief prompt
ctx = PromptContext(
    topic="why rest days make you fitter",
    channel="blog",
    audience="fitness_enthusiast",
    brand_voice=brand_voice,
    writing_rules=writing_rules,
    content_examples=content_examples,
    product_specs=product_specs,
    market_context=market_context,
    differentiators=differentiators,
)

brief_prompt = build_brief_prompt(ctx)
print("=== BRIEF GENERATION PROMPT (Chain-of-Thought) ===")
print(brief_prompt[:1200])
print("...")


=== BRIEF GENERATION PROMPT (Chain-of-Thought) ===
You are creating a content brief for a FitByte blog post about: "why rest days make you fitter"

TARGET AUDIENCE: Fitness Enthusiast
CHANNEL: blog

BRAND CONTEXT:
Brand Guidelines
FitByte — Brand Voice & Content Standards

Who We Are

FitByte is a fitness watch built for people who are serious about their health — but don't want a device that makes them feel like they're in a lab.

We sit at the intersection of performance technology and everyday life. Our wearers range from competitive athletes tracking race-day splits to busy professionals who want to close their rings and get on with their day. What they share is a belief that data — the right data, presented simply — can help them live and move better.

Our voice reflects that. Smart. Grounded. A little bit gritty.

Our Voice

Clear, not clinical
We deal in data — heart rate zones, sleep stages, recovery scores — but we never talk like a research paper. If a stat needs a sentence o

In [7]:
# Preview the full blog post generation prompt
blog_prompt = build_blog_post_prompt(ctx)
print("=== BLOG POST GENERATION PROMPT (Few-Shot + Contextual Placement) ===")
print(blog_prompt[:1500])
print("...")
print(f"\n[Total prompt length: {len(blog_prompt):,} chars]")


=== BLOG POST GENERATION PROMPT (Few-Shot + Contextual Placement) ===
Write a FitByte blog post about: "why rest days make you fitter"



BRAND VOICE & WRITING RULES (follow these exactly):
Brand Guidelines
FitByte — Brand Voice & Content Standards

Who We Are

FitByte is a fitness watch built for people who are serious about their health — but don't want a device that makes them feel like they're in a lab.

We sit at the intersection of performance technology and everyday life. Our wearers range from competitive athletes tracking race-day splits to busy professionals who want to close their rings and get on with their day. What they share is a belief that data — the right data, presented simply — can help them live and move better.

Our voice reflects that. Smart. Grounded. A little bit gritty.

Our Voice

Clear, not clinical
We deal in data — heart rate zones, sleep stages, recovery scores — but we never talk like a research paper. If a stat needs a sentence of context to be useful, 

## 4. LLM Integration

The `LLMClient` supports both OpenAI and Anthropic, with automatic retry on rate limits.
Run the cell below to initialize — it will auto-detect which API key you have set.


In [8]:
from llm_integration import get_llm_client

# Auto-detects OPENAI_API_KEY or ANTHROPIC_API_KEY from .env
llm = get_llm_client(provider="auto")
print(f"✓ LLM client initialized: {llm.__class__.__name__} / {llm.model}")


✓ LLM client initialized: OpenAIClient / gpt-4o-mini


## 5. Content Pipeline — Step by Step

Let's run each pipeline stage manually to see what's happening at each step.


### Step 1 + 2: Document & Monitor

In [9]:
from content_pipeline import ContentPipeline

TOPIC = "why rest days make you fitter"
CHANNEL = "blog"
AUDIENCE = "fitness_enthusiast"

pipeline = ContentPipeline(provider="auto", verbose=True)
context = pipeline.document(TOPIC, AUDIENCE)
monitor_report = pipeline.monitor(TOPIC, context)

print("\n=== MONITOR REPORT ===")
import json
print(json.dumps(monitor_report, indent=2))



[1/5] DOCUMENT — Loading knowledge bases...
  ✓ Brand voice: 7812 chars
  ✓ Product specs: 2503 chars
  ✓ Content examples: 2149 chars
  ✓ Market context: 1290 chars

[2/5] MONITOR — Analyzing topic...


RuntimeError: OpenAI API error after 1 attempt(s): Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-svcac***********************************************************************************************************************************************************n1QA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

### Step 3: Brief Generation (Chain-of-Thought)

In [10]:
content_brief = pipeline.brief(TOPIC, CHANNEL, AUDIENCE, context)
print("\n=== GENERATED BRIEF ===")
print(content_brief)



[3/5] BRIEF — Generating content brief...


RuntimeError: OpenAI API error after 1 attempt(s): Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-svcac***********************************************************************************************************************************************************n1QA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

### Step 4: Content Generation (Few-Shot + Contextual Placement)

In [ ]:
content = pipeline.publish(TOPIC, CHANNEL, AUDIENCE, context, content_brief)
print("\n" + "="*60)
print("FINAL BLOG POST")
print("="*60)
print(content)
print("="*60)
print(f"\nWord count: ~{len(content.split())} words")


## 6. Multi-Channel Generation

The same topic, adapted for different channels. Each uses a channel-specific prompt template.


In [ ]:
channels = ["instagram", "linkedin", "email_subject"]
topic_multi = "the link between HRV and how you feel in the morning"

print(f"Topic: '{topic_multi}'\n")

for channel in channels:
    ctx_ch = PromptContext(
        topic=topic_multi,
        channel=channel,
        brand_voice=brand_voice,
        writing_rules=writing_rules,
        content_examples=content_examples,
        product_specs=product_specs,
        market_context=market_context,
    )
    from prompt_templates import get_prompt_for_channel
    system_p, user_p = get_prompt_for_channel(ctx_ch)
    
    response = llm.generate(
        user_prompt=user_p,
        system_prompt=system_p,
        temperature=0.75,
        max_tokens=400,
    )
    
    print(f"{'='*60}")
    print(f"CHANNEL: {channel.upper()}")
    print(f"{'='*60}")
    print(response.content)
    print()


## 7. Uniqueness Demonstration

This is the key differentiator of the system. We run the **same topic** through:

1. **FitByte branded prompt**: Full system prompt + brand voice + few-shot examples + product specs + market context
2. **Generic prompt**: `"Write a short blog post about {topic} for a fitness watch brand."`

The comparison shows concretely why our system avoids "AI-Slop".


In [ ]:
from llm_integration import run_uniqueness_comparison
from prompt_templates import SYSTEM_PROMPT_WRITER, get_prompt_for_channel

comparison_topic = "sleep quality vs sleep quantity"

ctx_compare = PromptContext(
    topic=comparison_topic,
    channel="blog",
    brand_voice=brand_voice,
    writing_rules=writing_rules,
    content_examples=content_examples,
    product_specs=product_specs,
    market_context=market_context,
)

_, fitbyte_prompt = get_prompt_for_channel(ctx_compare)

comparison = run_uniqueness_comparison(
    topic=comparison_topic,
    fitbyte_prompt=fitbyte_prompt,
    fitbyte_system=SYSTEM_PROMPT_WRITER,
    llm=llm,
)


In [ ]:
# Display formatted comparison
print("=" * 60)
print("POST A — FITBYTE BRANDED SYSTEM")
print("=" * 60)
print(comparison["fitbyte_output"])

print("\n" + "=" * 60)
print("POST B — GENERIC PROMPT BASELINE")
print("=" * 60)
print(comparison["generic_output"])

print("\n" + "=" * 60)
print("ANALYSIS: Why Post A is more distinctive")
print("=" * 60)
print(comparison["analysis"])


### What Makes the FitByte Output Different

| Dimension | Generic Output | FitByte Branded Output |
|---|---|---|
| **Tone** | Generic motivational | FitByte voice: direct, human, no hype |
| **Product mentions** | Vague ("your watch") | Specific ("FitByte's 6-stage sleep tracking") |
| **Data specificity** | None | Exact specs from product KB |
| **Brand vocabulary** | Standard fitness language | FitByte-specific terms (Recovery Score, Body Battery) |
| **Writing rules** | Standard prose | No serial comma, no passive voice, em-dash style |
| **CTA style** | "Buy now" / hard sell | Quiet punchline, never a hard sell |
| **Content style** | Generic structure | Matches real FitByte blog format (150-250 words, 2nd person) |


## 8. Batch Generation

Generate multiple pieces automatically — useful for content calendar planning.


In [ ]:
BATCH_TOPICS = [
    {"topic": "how training load prevents injuries", "channel": "blog", "audience": "performance_athlete"},
    {"topic": "winter running motivation", "channel": "blog", "audience": "fitness_enthusiast"},
    {"topic": "the 5 metrics worth checking daily", "channel": "instagram", "audience": "general"},
]

print(f"Running batch: {len(BATCH_TOPICS)} content pieces\n")
batch_results = pipeline.run_batch(BATCH_TOPICS)

print("\n" + "="*60)
print("BATCH RESULTS SUMMARY")
print("="*60)
for i, result in enumerate(batch_results, 1):
    print(f"\n[{i}] {result['topic'][:50]}")
    print(f"    Channel: {result['channel']}")
    print(f"    Words: ~{len(result['content'].split())}")
    print(f"    Preview: {result['content'][:100]}...")


## 9. System Architecture

```
fitbyte-ai-content-creator/
├── main.py                          # CLI entry point
├── src/
│   ├── document_processor.py        # Markdown parsing & text extraction
│   ├── knowledge_base.py            # KB management (primary + secondary)
│   ├── prompt_templates.py          # Advanced prompt engineering templates
│   ├── content_pipeline.py          # Document→Monitor→Brief→Publish→Iterate
│   └── llm_integration.py           # OpenAI/Anthropic API wrapper
├── knowledge_base/
│   ├── primary/
│   │   ├── fitbyte_brand_guidelines.md
│   │   ├── fitbyte_product_specs.md
│   │   └── past_content/
│   │       └── fitbyte_content_examples.md
│   └── secondary/
│       ├── market_trends.md
│       └── competitor_analysis.md
├── outputs/                         # Generated content saved here
├── requirements.txt
├── .env.example
└── README.md
```

### Pipeline Flow
```
Knowledge Bases (Markdown)
        │
        ▼
[1] DOCUMENT — document_processor.py parses .md files
        │         knowledge_base.py provides structured access
        ▼
[2] MONITOR — LLM analyzes topic for brand fit + market relevance
        │
        ▼
[3] BRIEF — Chain-of-thought prompt → structured content brief
        │       (angle, hook, core insight, CTA)
        ▼
[4] PUBLISH — Few-shot + contextual placement prompt
        │        injects brand voice, examples, product specs
        ▼
[5] ITERATE — Human review: approve / edit / regenerate
        │
        ▼
    outputs/ (saved as .txt)
```


## 10. CLI Usage

The system also runs as a command-line tool:

```bash
# Interactive blog post
python main.py --topic "why your resting heart rate matters" --channel blog

# Instagram caption (auto-save)
python main.py --topic "winter training" --channel instagram --auto

# LinkedIn post for health professionals
python main.py --topic "stress and recovery at work" --channel linkedin --audience health_professional

# Uniqueness comparison
python main.py --topic "sleep quality" --compare

# Full batch run
python main.py --batch

# Use Anthropic instead of OpenAI
python main.py --topic "overtraining signs" --provider anthropic
```


## 11. Gradio Interface

The Gradio app (`app.py`) provides a full browser-based UI for the content pipeline.

**Features:**
- Topic, audience, channel and custom instructions inputs
- Live editable output box with word count
- AI refinement — type an instruction to rewrite a specific part
- Approve & Download — exports the final post as a `.txt` file


In [ ]:
!pip install gradio
import subprocess
import threading
import time
import gradio as gr

# Launch app.py as a background process
_proc = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for Gradio to start, then print the URL
def _tail_logs(proc):
    for line in proc.stdout:
        text = line.decode()
        print(text, end="")
        if "Running on" in text or "localhost" in text:
            break

threading.Thread(target=_tail_logs, args=(_proc,), daemon=True).start()
time.sleep(4)   # give Gradio a moment to bind

print("\n✓ Gradio app is running.")
print("  Open in your browser → http://localhost:7860")
print("\n  (Run the cell below to embed it inline, or just use the link above.)")


  Using cached gradio-6.14.0-py3-none-any.whl.metadata (17 kB)
  Using cached brotli-1.2.0-cp313-cp313-macosx_10_13_universal2.whl.metadata (6.1 kB)
  Using cached fastapi-0.136.1-py3-none-any.whl.metadata (28 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached python_multipart-0.0.27-py3-none-any.whl.metadata (2.1 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached starlette-1.0.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached uvicorn-0.46.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
Using cached gradi

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✓ Gradio app is running.
  Open in your browser → http://localhost:7860

  (Run the cell below to embed it inline, or just use the link above.)


/Users/markusvonaschoff/Desktop/ironhack_Bootcamp/Deliverables/Week 3/Lab 1/ai-content-creator/0_fitbyte-ai-content-creator_v3/app.py:295: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=FITBYTE_CSS, title="FitByte Content Creator") as demo:
Traceback (most recent call last):
  File "/Users/markusvonaschoff/Desktop/ironhack_Bootcamp/Deliverables/Week 3/Lab 1/ai-content-creator/0_fitbyte-ai-content-creator_v3/app.py", line 355, in <module>
    output_box = gr.Textbox(
        label="",
    ...<4 lines>...
        elem_classes=["output-textarea"],
    )
  File "/opt/miniconda3/lib/python3.13/site-packages/gradio/component_meta.py", line 194, in wrapper
    return fn(self, **kwargs)
TypeError: Textbox.__init__() got an unexpected keyword argument 'show_copy_button'


In [15]:
# Embed the Gradio app inline inside the notebook
from IPython.display import IFrame, display
display(IFrame(src="http://localhost:7860", width="100%", height=820))


---
## Summary

**What we built:**
- A full Python content creation system for FitByte fitness watches
- Two knowledge bases (primary brand + secondary research) parsed from markdown
- 5-stage pipeline with human-in-the-loop review
- Advanced prompt engineering using role prompting, few-shot examples, chain-of-thought, and contextual placement
- Multi-channel support (blog, Instagram, LinkedIn, email subject lines)
- Uniqueness validation — side-by-side comparison vs generic prompts
- CLI interface + batch mode

**Why it avoids AI-Slop:**
Every prompt is anchored to FitByte's actual brand voice, real writing rules, genuine product specs, and real content examples. The model cannot produce generic output because generic elements have been replaced with specific, proprietary context at every stage.
